# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research question:** Which pre-decline signals — content staleness, click-through rate
relative to search position, and word count — are actually associated with a page losing
search visibility, measured on data available *before* the decline happens?

**Decision this supports:** which pages a content/SEO reviewer with limited time should look
at first. Not an automated fix — a prioritization aid.

**Real-world motivation:** FlyRank's own March 2026 research paper found that refresh timing
is "one of the strongest measured levers available" (3.2x health boost, 57x more impressions
on refreshed 365+ day content) — but that only helps once a reviewer knows which pages to
refresh first. This capstone answers that upstream question directly.

In [1]:
print("Lane: Ranking Signal Analysis")
print("Decision supported: which pages a reviewer should check first for possible decline")


Lane: Ranking Signal Analysis
Decision supported: which pages a reviewer should check first for possible decline


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank pseudonymized search-performance warehouse (Hugging Face,
`FlyRank/internship-warehouse`) — `dim_clients`, `dim_content`, and
`fact_content_daily_performance`, joined and filtered to content with `imp_prev30 >= 100`.

**Coverage:** 111,247 of 519,606 total content items (21.4% of the portfolio) across 104
clients.

**Windows:** `prev30` (days 31-60 back from the snapshot) builds every feature; `last30`
(most recent 30 days) is used only to compute the label, never as a feature.

**Label:** declining = impressions drop more than 20% from `prev30` to `last30`. Checked
against FlyRank's own published 10% threshold — the two definitions disagree on 6,563 pages
(5.9% of the working set).

**Excluded, with reasons:**
- `imp_last30`, `clk_last30`, `pos_last30`, `trend_direction`, `trend_pct` — inside the label
  window, direct leakage (confirmed via correlation: -0.14 to +0.18 against the label, well
  above any legitimate pre-decline feature).
- `fact_content_query_90d` columns (`rare_share`, `top_query_share`) — fixed 90-day window
  overlaps the label period with no safe way to split it, excluded entirely rather than used
  with a caveat.

**Public-safe:** all identifiers are pseudonymous (`client_hash_id`, `content_hash_id`); no
client names, URLs, or raw queries appear anywhere in this notebook or the deployed paper.

In [2]:
# Summary numbers carried over from the validated Week 3-7 notebooks
# (source of truth — not re-derived here, this capstone compiles them).
data_summary = {
    'total_content_items': 519606,
    'covered_content_items': 111247,
    'coverage_pct': 111247 / 519606,
    'clients': 104,
    'label_threshold_project': 0.20,
    'label_threshold_flyrank_paper': 0.10,
    'rows_where_thresholds_disagree': 6563,
}
print(data_summary)


{'total_content_items': 519606, 'covered_content_items': 111247, 'coverage_pct': 0.21409875944465614, 'clients': 104, 'label_threshold_project': 0.2, 'label_threshold_flyrank_paper': 0.1, 'rows_where_thresholds_disagree': 6563}


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Baseline rule:** flag a page when `days_since_last_update > 90`, OR when its `ctr_prev30`
sits below its position tier's median (using a zero-clicks check instead for the two tiers
whose median CTR is exactly 0). Reason codes: `STALE`, `WEAK_CTR`, `STALE+WEAK_CTR`, `NONE`.

**Models:** Logistic Regression (interpretable coefficients — this study's purpose is finding
which signals to trust) and Random Forest with permutation importance (a nonlinear cross-check).

**Features used:** `imp_prev30`, `clk_prev30`, `pos_prev30`, `ctr_prev30`,
`days_since_last_update`, `word_count`, `char_count`, `content_type`, `main_intent` — all
measurable before the label window.

**Validation design:** `GroupShuffleSplit` on `client_hash_id`, confirmed to produce zero
client overlap between train and test. Chosen because pages from the same client share
templates and publishing habits — a random split would let a model see near-duplicate patterns
from the same client on both sides, inflating its apparent performance.

**Leakage checks:** label-window columns confirmed to correlate 0.14-0.18 against the label
(vs. 0.02-0.16 for the safe feature set, with `ctr_prev30` as the one safe feature worth a
second look given its comparably high 0.16 correlation).

In [3]:
safe_features = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                 'days_since_last_update', 'word_count', 'char_count',
                 'content_type', 'main_intent']
excluded_features = ['imp_last30', 'clk_last30', 'pos_last30', 'trend_direction', 'trend_pct']

print("Features used:", safe_features)
print("\nExcluded (leakage):", excluded_features)


Features used: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30', 'days_since_last_update', 'word_count', 'char_count', 'content_type', 'main_intent']

Excluded (leakage): ['imp_last30', 'clk_last30', 'pos_last30', 'trend_direction', 'trend_pct']


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

No single approach wins outright. The baseline rule is the most **precise** (0.735) — when it
flags a page, it's right most often, which matters for a queue a reviewer must trust
repeatedly. Logistic Regression has the best F1 (0.771) and recall (0.911) but more false
alarms. Random Forest, despite being the most complex model, is the **weakest** of the three on
F1 (0.695).

**Why the split mattered more than the model:** under a naive random split, Random Forest's F1
was 0.790 — it would have looked like the clear winner. Under the honest client-grouped split,
it dropped to 0.695, a 0.095 inflation — enough to flip it from best model to weakest. Logistic
Regression barely moved (0.781 -> 0.771), since its lower capacity gave it far less room to
memorize client-specific patterns.

In [4]:
import pandas as pd

# Honest client-grouped split results
results_honest = pd.DataFrame({
    'approach': ['Baseline rule', 'Logistic Regression', 'Random Forest'],
    'precision': [0.735123, 0.668108, 0.705061],
    'recall':    [0.548398, 0.911104, 0.685136],
    'f1':        [0.628178, 0.770911, 0.694955],
    'auc_roc':   [None, 0.548020, 0.599703],
})
print(results_honest.to_string(index=False))

print()

# Naive random split vs honest grouped split (Week 6 finding)
split_comparison = pd.DataFrame({
    'model': ['Logistic Regression', 'Random Forest'],
    'f1_random_split': [0.781425, 0.789583],
    'f1_grouped_split': [0.770911, 0.694955],
})
split_comparison['inflation'] = split_comparison['f1_random_split'] - split_comparison['f1_grouped_split']
print(split_comparison.to_string(index=False))


           approach  precision   recall       f1  auc_roc
      Baseline rule   0.735123 0.548398 0.628178      NaN
Logistic Regression   0.668108 0.911104 0.770911 0.548020
      Random Forest   0.705061 0.685136 0.694955 0.599703

              model  f1_random_split  f1_grouped_split  inflation
Logistic Regression         0.781425          0.770911   0.010514
      Random Forest         0.789583          0.694955   0.094628


## 5. Limitations

*What this work cannot claim.*

- **The label is a proxy, not a proven outcome** — describes the current 30-day window, not a
  guaranteed future trajectory for any individual page.
- **Coverage stops at 21.4% of the portfolio** — the 100-impression floor excludes the long
  tail of low-traffic pages.
- **Modest discriminative power** — AUC-ROC of 0.55-0.60 sits closer to chance than strong
  separation; this is a prioritization aid, not a confident prediction.
- **Logistic Regression's feature ranking is scale-confounded** — numeric features were never
  standardized before fitting, so its coefficient magnitudes aren't fairly comparable across
  features of very different natural scale.
- **Observational, not causal, throughout** — every relationship is an association measured in
  this portfolio, not proof that any action causes a change in performance.
- **A global time anchor, not a per-client one** — clients whose data ends earlier are
  proportionally underrepresented.

In [5]:
coverage_pct = 111247 / 519606
print(f"Portfolio coverage: {coverage_pct:.1%} -- findings apply to this slice only")


Portfolio coverage: 21.4% -- findings apply to this slice only


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

The baseline's reason code drives the primary ranking, not the model — it is the most precise
and most explainable of the three approaches. The model's confidence score is used only as a
secondary, within-group ordering signal.

| Reason code | Action | Meaning |
|---|---|---|
| `STALE+WEAK_CTR` | Refresh, highest priority | 90+ days stale AND weak CTR for its position |
| `STALE` | Refresh | 90+ days stale, CTR looks normal |
| `WEAK_CTR` | Monitor | Fresh, but weak CTR for its position |
| `NONE` | Leave | No warning signs |

**Never automate:** no auto-editing/redirecting/unpublishing on this score alone; no using it
as a judgment of a writer's performance; no applying it to clients or pages outside the
coverage above; no treating a flag as confirmed decline without a human check for context the
model can't see (evergreen intent, inherently low-CTR traffic, deleted/unpublished pages).

In [6]:
reason_code_counts = {
    'WEAK_CTR': 50144,
    'NONE': 45489,
    'STALE+WEAK_CTR': 8930,
    'STALE': 6684,
}
print(reason_code_counts)
print(f"Total: {sum(reason_code_counts.values())}")
print(f"Rows removed by no-go filter (deleted/unpublished): 160")


{'WEAK_CTR': 50144, 'NONE': 45489, 'STALE+WEAK_CTR': 8930, 'STALE': 6684}
Total: 111247
Rows removed by no-go filter (deleted/unpublished): 160


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

The deployed page's central chart (naive-vs-honest split F1 gap) is built as inline SVG
directly in `docs/index.html` — no external image dependency, so it can never break from a
broken relative path. The results and recommendation tables are likewise written directly as
HTML tables from the same numbers reproduced in Section 4 and Section 6 above.

A backup PNG of the same split-comparison chart was also exported in Week 7
(`work/figures/honest_split_before_after.png`) and `work/outputs/playbook_metrics.json` holds
the underlying reference numbers — both committed, both available if the page ever needs a
static image version instead of the inline SVG.

In [7]:
import os
print("Figure committed:", os.path.exists('work/figures/honest_split_before_after.png') if os.path.exists('work') else 'check in repo')
print("Metrics JSON committed:", os.path.exists('work/outputs/playbook_metrics.json') if os.path.exists('work') else 'check in repo')
print("Deployed page: docs/index.html -- inline SVG chart, no external image dependency")


Figure committed: check in repo
Metrics JSON committed: check in repo
Deployed page: docs/index.html -- inline SVG chart, no external image dependency


## 8. 5-Minute Demo Outline

**1. Question (30s)** — A reviewer can't check every page. Which pre-decline signals
actually tell you what to check first?

**2. Method (1 min)** — A simple, readable baseline rule (stale content + weak
click-through rate) vs. two trained models (Logistic Regression, Random Forest), all
validated on the same client-grouped split so no client's pages ever appear in both training
and testing.

**3. One chart (1.5 min)** — The honest-vs-naive split chart. Under a naive random split,
Random Forest looks like the best model (F1 0.790). Under the honest client-grouped split, it
drops to 0.695 -- the weakest of the three. That gap is the whole argument for why the split
mattered more than the model.

**4. One honest result (1 min)** — The baseline rule has the best precision (0.735) of all
three approaches. It isn't the fanciest tool, but it's the one a reviewer can trust the most
often when it flags something.

**5. One recommendation (1 min)** — The final ranked action queue: reason codes (STALE,
WEAK_CTR, STALE+WEAK_CTR) drive the ranking, the model adds a secondary confidence score, and
a documented no-go list stops the system from being used for anything it wasn't validated on.

## 9. Shareable Cuts

**Social post:**

Tested whether a simple, readable rule could out-predict a full ML model for flagging
declining SEO pages -- on 111,247 real content pages. It won on precision (0.735 vs. Random
Forest's 0.705), once I removed a leakage issue most quick benchmarks miss: pages from the
same client sneaking into both training and test data. One chart tells the whole story ->
[Paper Link](https://hassannawaz14.github.io/FlyRank-ML-Internship/)

**Employer 3-sentencer:**

I built a validated early-warning system that ranks SEO content pages by which pre-decline
signals actually predict falling search performance. It runs on 111,247 real pages from
FlyRank's search-performance warehouse, using a client-grouped validation split to catch a
leakage issue that silently inflates most naive model benchmarks. The result: a simple,
explainable rule beat a more complex model on precision, and the same leakage check revealed
the complex model's apparent edge was entirely a testing artifact -- a finding that reshaped
how I approached model selection for the rest of the project.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.